# 🚀 Direct access to EarthScope S3 cloud storage: boto3


**Author:** Yiyu Ni  
**Goal:** This notebook demonstrates how to access earthscope S3 cloud stoage and load miniSEED data.  

---

In [2]:
import io, os
import obspy
import boto3

from parameters import ES_S3_ACCESS_POINT
from earthscope_sdk import EarthScopeClient

## 1. Get temporary credential through EarthScope Client

In [3]:
client = EarthScopeClient()
cred = client.user.get_aws_credentials()
print(f"Credential expires at {cred.expiration} UTC")

Credential expires at 2025-10-14 06:45:34+00:00 UTC


## 2. Prepare S3 access
The EarthScope credential object **cred** provides a temporary AWS access key ID, access key, and session token, all valid for an hour.

In [4]:
session = boto3.Session(
    aws_access_key_id=cred.aws_access_key_id,
    aws_secret_access_key=cred.aws_secret_access_key,
    aws_session_token=cred.aws_session_token,
)

s3_client = session.client("s3")

### 3. Access the bucket through the S3 access point
The **ES_S3_ACCESS_POINT** provides the entry point to the S3 bucket. Instead of directly accessing an S3 bucket, additional authentication setting at finer level could be configured through this layer. The access point is defined [here](./parameters.py).

With boto3, we would need to use S3-style methods to browse the bucket, e.g., `list_objects`. 

In [5]:
obj = s3_client.list_objects(Bucket=ES_S3_ACCESS_POINT, 
                             Prefix="miniseed/CC/2006/001/")
obj

{'ResponseMetadata': {'RequestId': '243a1440-632d-4630-bb73-181165dbb739',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '243a1440-632d-4630-bb73-181165dbb739',
   'server': 'AmazonS3',
   'x-amz-bucket-region': 'us-east-2',
   'x-amz-request-id': '43GTX5NT7EJRM137',
   'content-type': 'application/xml',
   'x-amz-id-2': '2nkdSRJRXJjmAba6NHESuA/FtRpuSaL36qz5b7zIPxQR2BcBBpzAuRle+PzcWxZJL2mEhNg1Z4s=',
   'date': 'Tue, 14 Oct 2025 06:04:53 GMT',
   'transfer-encoding': 'chunked',
   'connection': 'keep-alive'},
  'RetryAttempts': 0},
 'IsTruncated': False,
 'Marker': '',
 'Contents': [{'Key': 'miniseed/CC/2006/001/JRO.CC.2006.001#1',
   'LastModified': datetime.datetime(2024, 3, 7, 6, 43, 45, tzinfo=tzlocal()),
   'ETag': '"caf671881f20d83c273fca830b267343"',
   'Size': 18982400,
   'StorageClass': 'INTELLIGENT_TIERING',
   'Owner': {'ID': 'f39d8aa4d562bac789be09593fab9ed63c22b031e4dc0b8c59e5f1eac9ad38dc'}},
  {'Key': 'miniseed/CC/2006/001/NED.CC.2006.001#1',
   'LastModi

The output of `list_objects` is more like a JSON style response and contains more metadata (e.g., file size, storage class, etc.).

---
Here, we can use `fs.get_object` to read one file. Finally, we parse the bytes into `obspy.Stream`.

In [21]:
obj = s3_client.get_object(Bucket=ES_S3_ACCESS_POINT, 
                           Key="miniseed/CC/2006/001/STD.CC.2006.001#1")
obj

{'ResponseMetadata': {'RequestId': '49c81a01-5fb3-4c14-ad5d-3524bfee2dcc',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '49c81a01-5fb3-4c14-ad5d-3524bfee2dcc',
   'x-amz-request-id': '49c81a01-5fb3-4c14-ad5d-3524bfee2dcc',
   'content-type': 'application/vnd.fdsn.mseed',
   'x-amz-id-2': '{id-2}',
   'date': 'Tue, 14 Oct 2025 06:12:05 GMT',
   'content-length': '34823680',
   'connection': 'keep-alive'},
  'RetryAttempts': 0},
 'ContentLength': 34823680,
 'ContentType': 'application/vnd.fdsn.mseed',
 'Metadata': {},
 'Body': <botocore.response.StreamingBody at 0x7f86ce3546a0>}

In [22]:
bytes = obj["Body"].read()

In [23]:
buff = io.BytesIO(bytes)

In [24]:
obspy.read(buff)

5 Trace(s) in Stream:
CC.STD..BHE | 2006-01-01T00:00:00.000000Z - 2006-01-01T20:05:14.980000Z | 50.0 Hz, 3615750 samples
CC.STD..BHE | 2006-01-01T20:05:20.020000Z - 2006-01-01T23:59:59.980000Z | 50.0 Hz, 703999 samples
CC.STD..BHN | 2006-01-01T00:00:00.000000Z - 2006-01-01T23:59:59.980000Z | 50.0 Hz, 4320000 samples
CC.STD..BHZ | 2006-01-01T00:00:00.000000Z - 2006-01-01T20:05:14.980000Z | 50.0 Hz, 3615750 samples
CC.STD..BHZ | 2006-01-01T20:05:20.020000Z - 2006-01-01T23:59:59.980000Z | 50.0 Hz, 703999 samples